In [ ]:
import cv2
import mediapipe as mp
import os
print("cv2 version:", cv2.__version__)
print("mediapipe version:", mp.__version__)


In [ ]:
mp_hands = mp.solutions.hands
hands = mp_hands.Hands(static_image_mode=False, max_num_hands=1, min_detection_confidence=0.7)
mp_draw = mp.solutions.drawing_utils

In [ ]:
cap = cv2.VideoCapture(0)

In [7]:
import cv2
import mediapipe as mp
import os
import time
import webbrowser

mp_hands = mp.solutions.hands
hands = mp_hands.Hands(static_image_mode=False, max_num_hands=1, min_detection_confidence=0.7)
mp_draw = mp.solutions.drawing_utils
cap = cv2.VideoCapture(0)

last_finger_count = -1
last_action_time = time.time()

def detect_fingers(hand_landmarks):
    tips_ids = [4, 8, 12, 16, 20]
    fingers = []
    landmarks = hand_landmarks.landmark
    if landmarks[tips_ids[0]].x < landmarks[tips_ids[0] - 1].x:
        fingers.append(1)
    else:
        fingers.append(0)
    for id in range(1, 5):
        if landmarks[tips_ids[id]].y < landmarks[tips_ids[id] - 2].y:
            fingers.append(1)
        else:
            fingers.append(0)
    return sum(fingers)

running = True

while running:
    ret, frame = cap.read()
    if not ret:
        break
    frame = cv2.flip(frame, 1)
    rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    result = hands.process(rgb_frame)

    if result.multi_hand_landmarks:
        for hand_landmarks in result.multi_hand_landmarks:
            mp_draw.draw_landmarks(frame, hand_landmarks, mp_hands.HAND_CONNECTIONS)
            finger_count = detect_fingers(hand_landmarks)
            cv2.putText(frame, f"Fingers Up: {finger_count}", (10, 40), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)

            if finger_count != last_finger_count and (time.time() - last_action_time > 1.5):
                last_finger_count = finger_count
                last_action_time = time.time()

                if finger_count == 1:
                    os.system("notepad")
                elif finger_count == 2:
                    os.system("calc")
                elif finger_count == 3:
                    webbrowser.open("https://lakshya-chalana-portfolio.netlify.app/")
                elif finger_count == 4:
                    webbrowser.open("https://www.google.com")
                elif finger_count == 5:
                    running = False

    cv2.imshow("Finger Detection", frame)
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()
